# 4. The quantum KNN, from the maths up

This is the heart of the course's section 17.  We will:

1. compute the classical KNN;
2. work out what the quantum circuit computes;
3. show the two are the same number.


In [ ]:
import math
import sys
from pathlib import Path

# Make the shared qc101 library importable, whether the notebook is opened
# from the repository root or from inside the notebooks/ directory.
_here = Path.cwd().resolve()
for _candidate in (_here / "labs" / "common",
                   _here.parent / "labs" / "common",
                   _here / ".." / "labs" / "common"):
    if (_candidate / "qc101").is_dir():
        sys.path.insert(0, str(_candidate.resolve()))
        break
else:
    raise SystemExit("Could not find the qc101 library -- run this notebook "
                     "from the repository root or from notebooks/.")

from qc101 import boolean, complexnum, linalg, photon, probability, quantum, viz
from qc101.labkit import Lab

print("qc101 library ready")


In [ ]:
import math

TRAINING = [[0.9798, 0.2, 1], [0.3, 0.954, 0]]
QUERY = [0.85, 0.5268]

def normalize(v):
    n = math.sqrt(sum(x * x for x in v))
    return [x / n for x in v]

## Step 1: the classical KNN

Normalise, compute squared distances, turn distances into weights, and
normalise the weights into probabilities.


In [ ]:
vectors = [normalize(t[:2]) for t in TRAINING]
q = normalize(QUERY)

weights = []
for v in vectors:
    d2 = sum((qi - vi) ** 2 for qi, vi in zip(q, v))
    weights.append((d2, 1 - 0.25 * d2))

total = sum(w for _, w in weights)
classical = {t[2]: 0.0 for t in TRAINING}
for t, (_, w) in zip(TRAINING, weights):
    classical[t[2]] += w / total

for t, (d2, w) in zip(TRAINING, weights):
    print(f"class {t[2]}:  d^2 = {d2:.6f}   w = {w:.6f}")
print("classical probabilities:", classical)

## Step 2: the interference amplitudes

At a slot holding a training component $v_i$ and the query component $q_i$,
the Hadamard on the 'training vs query' qubit leaves the amplitude

$$\frac{v_i + q_i}{2\sqrt2}$$

so the probability of that slot in the training half is proportional to
$(v_i + q_i)^2$.


In [ ]:
for label, v in zip(("training 0", "training 1"), vectors):
    interference = sum((vi + qi) ** 2 for vi, qi in zip(v, q))
    d2 = sum((qi - vi) ** 2 for qi, vi in zip(q, v))
    classical_w = 1 - 0.25 * d2
    print(f"{label}:  sum (v_i + q_i)^2 = {interference:.6f}   "
          f"4 * w = {4 * classical_w:.6f}   equal: {abs(interference - 4*classical_w) < 1e-12}")

## Step 3: the class probability

Sum the interference amplitudes per class, then normalise.


In [ ]:
quantum = {0: 0.0, 1: 0.0}
for v, t in zip(vectors, TRAINING):
    quantum[t[2]] += sum((vi + qi) ** 2 for vi, qi in zip(v, q))

total = sum(quantum.values())
quantum = {k: v / total for k, v in quantum.items()}

for k in sorted(classical):
    print(f"class {k}:  classical {classical[k]:.6f}   quantum {quantum[k]:.6f}")

## Step 4: the real circuit

The same thing, done with Qiskit.  If Qiskit is installed the next cell
runs; otherwise it prints a note.


In [ ]:
try:
    import numpy as np
    from qiskit import QuantumCircuit, transpile
    from qiskit.quantum_info import Statevector
    from qiskit_aer import AerSimulator

    amplitudes = [0.0] * 16

    def index(q0, q1, q2, q3):
        return q0 + 2 * q1 + 4 * q2 + 8 * q3

    for v, t in zip(vectors, TRAINING):
        class_bit = 1 if t[2] == TRAINING[0][2] else 0
        amplitudes[index(class_bit, 0, 0, 0)] = v[0] / 2
        amplitudes[index(class_bit, 1, 0, 0)] = v[1] / 2
    for class_bit in (1, 0):
        amplitudes[index(class_bit, 0, 0, 1)] = q[0] / 2
        amplitudes[index(class_bit, 1, 0, 1)] = q[1] / 2

    qc = QuantumCircuit(4, 2)
    qc.initialize(amplitudes)
    qc.h(3)
    qc.measure(3, 0)
    qc.measure(0, 1)

    counts = (AerSimulator().run(transpile(qc, AerSimulator()), shots=20000)
              .result().get_counts())
    q3zero = counts.get("00", 0) + counts.get("10", 0)
    p1 = counts.get("10", 0) / q3zero
    print("counts:", counts)
    print(f"quantum P(class 1) = {p1:.4f}   classical = {classical[1]:.4f}")
except ImportError as exc:
    print("Qiskit is not installed:", exc)